# JPQ Pseudo-Relevance Feedback

This notebook evaluates the PQ PRF variants in `pyterrier_dr.jpq.prf` on the MS MARCO passage TREC DL 2019 benchmark, using the same JPQ TCT index as `jpq-tct-replication.ipynb`.

It compares four feedback methods and sweeps the interpolation weight `alpha`:

- `PQPrfTF`
- `PQPrfTFIDF`
- `PQPrfProbabilisticTFIDF`
- `PQPrfKLTFIDF`

Install the local `pyterrier_dr` checkout with the JPQ extras, plus FAISS and a compatible Transformers version. Run this notebook from the `examples/` directory.

In [ ]:
%pip install -q -e '..[jpq]'
%pip install -q faiss-cpu "transformers==4.57.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyterrier as pt
import pyterrier_dr
import pyterrier_dr.jpq as jpq
from pyterrier.measures import nDCG
from pyterrier_dr.jpq.prf import (
    PQPrfTF,
    PQPrfTFIDF,
    PQPrfProbabilisticTFIDF,
    PQPrfKLTFIDF,
)

## Load the JPQ index and query encoder

This is the same TCT-ColBERT JPQ index used in the replication notebook.

In [ ]:
index = jpq.JPQIndex.from_hf(
    "jpq-repro/msmarco-passage-train__tct_colbert__faiss2opq__M96_nbits8__ps159744__neg200__ibn__lr__"
)
model = pyterrier_dr.TctColBert.hnp()
model.model = model.model.from_pretrained(index.path)

## First-pass retrieval and PRF pipeline

We retrieve a short first-pass list, load each document's PQ codes, and then run a PRF transformer before the final retrieval.

In [ ]:
query_encoder = model.query_encoder()

FIRST_PASS_K = 100   # documents retrieved for feedback
PRF_K = 10           # pseudo-relevant documents used by PRF
FINAL_K = 1000       # final result list

first_pass = (
    query_encoder
    >> index.retriever_pq(topk=FIRST_PASS_K)
    >> index.vec_loader()
    >> query_encoder  # re-attach query_vec after retrieval
)

The JPQ index uses OPQ. The query encoder produces vectors in the original space, while the PQ centroids live in the rotated space, so PRF is run in the rotated space and the feedback vector is rotated back afterwards.

In [ ]:
class OPQPrf(pt.Transformer):
    """Runs a PQ PRF transformer in the OPQ-rotated space."""
    def __init__(self, prf, opq):
        self.prf = prf
        self.opq = np.asarray(opq)

    def transform(self, inp):
        inp = inp.assign(
            query_vec=[np.asarray(v) @ self.opq for v in inp['query_vec']]
        )
        out = self.prf(inp)
        return out.assign(
            query_vec=[np.asarray(v) @ self.opq.T for v in out['query_vec']]
        )


def make_prf_pipeline(prf):
    if index.opq is not None:
        prf = OPQPrf(prf, index.opq)
    return first_pass >> prf >> index.retriever_pq(topk=FINAL_K)

## Alpha sweep

We sweep `alpha` from 0 (pure feedback) to 1 (no feedback) for each PRF variant.

In [ ]:
# Compute collection code frequencies once and reuse them for every variant.
collection_code_freq = np.stack([
    np.bincount(index.codes[:, m], minlength=index.meta.Ks)
    for m in range(index.meta.M)
])

variants = {
    'TF': PQPrfTF,
    'TF-IDF': PQPrfTFIDF,
    'Probabilistic TF-IDF': PQPrfProbabilisticTFIDF,
    'KL TF-IDF': PQPrfKLTFIDF,
}

alphas = np.round(np.arange(0.0, 1.01, 0.1), 2)

pipelines = []
names = []
params = []
for name, cls in variants.items():
    for alpha in alphas:
        pipelines.append(
            make_prf_pipeline(cls(
                centroids=index.dvecs,
                collection_code_freq=collection_code_freq,
                n_docs=len(index),
                alpha=float(alpha),
                k=PRF_K,
            ))
        )
        names.append(f'{name} (alpha={alpha:.1f})')
        params.append((name, float(alpha)))

Run the sweep on TREC DL 2019.

In [ ]:
dataset = pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged")
results = pt.Experiment(
    pipelines,
    dataset.get_topics(),
    dataset.get_qrels(),
    [nDCG@10],
    names=names,
    validate='ignore',
)
results

Convert the results to a tidy frame and show the best alpha for each variant.

In [ ]:
sweep = pd.DataFrame([
    {'variant': variant, 'alpha': alpha, 'nDCG@10': results.loc[name, 'nDCG@10']}
    for name, (variant, alpha) in zip(names, params)
])

best = sweep.loc[sweep.groupby('variant')['nDCG@10'].idxmax()]
best

## Plot alpha vs nDCG@10

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for variant, group in sweep.groupby('variant'):
    group = group.sort_values('alpha')
    ax.plot(group['alpha'], group['nDCG@10'], marker='o', label=variant)

ax.set_xlabel('alpha (weight on original query)')
ax.set_ylabel('nDCG@10')
ax.set_title('PQ PRF alpha sweep on TREC DL 2019')
ax.set_xticks(alphas)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## PRF depth sweep

Now we scan the number of pseudo-relevant documents `k`, using each variant's best `alpha` from the sweep above.

In [ ]:
best_alpha = best.set_index('variant')['alpha'].to_dict()
k_values = [1, 3, 5, 10, 20, 50, 100]

k_pipelines = []
k_names = []
k_params = []
for name, cls in variants.items():
    alpha = float(best_alpha[name])
    for k in k_values:
        k_pipelines.append(
            make_prf_pipeline(cls(
                centroids=index.dvecs,
                collection_code_freq=collection_code_freq,
                n_docs=len(index),
                alpha=alpha,
                k=k,
            ))
        )
        k_names.append(f'{name} (k={k}, alpha={alpha:.1f})')
        k_params.append((name, k))

Run the PRF depth sweep on TREC DL 2019.

In [ ]:
depth_results = pt.Experiment(
    k_pipelines,
    dataset.get_topics(),
    dataset.get_qrels(),
    [nDCG@10],
    names=k_names,
    validate='ignore',
)
depth_results

Convert the depth results to a tidy frame.

In [ ]:
depth_sweep = pd.DataFrame([
    {'variant': variant, 'k': k, 'nDCG@10': depth_results.loc[name, 'nDCG@10']}
    for name, (variant, k) in zip(k_names, k_params)
])

## Plot number of PRF documents vs nDCG@10

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for variant, group in depth_sweep.groupby('variant'):
    group = group.sort_values('k')
    ax.plot(group['k'], group['nDCG@10'], marker='o', label=variant)

ax.set_xlabel('number of PRF documents (k)')
ax.set_ylabel('nDCG@10')
ax.set_title('PQ PRF depth sweep on TREC DL 2019')
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

`alpha=1` corresponds to no feedback; lower values place more weight on the PQ feedback vector.